# TrakrAI GPU worker for Google Colab

This notebook lends a Colab GPU to **your factory's** model training in TrakrAI. It only runs jobs for the factory that approved it, and it stops when you stop this notebook.

**Before you start**

1. This notebook asks Colab for a **T4 GPU** when it opens. If Colab shows a different runtime, choose **Runtime → Change runtime type → T4 GPU** and **Save**.
2. Use a Colab account with a paid plan or a positive compute-unit balance. Colab's terms do not allow running workers like this one on the free tier.
3. You, or someone who can approve for you, need a TrakrAI login with the model admin permission for the factory.

**Run it**

Check that `TRAKRAI_URL` is your TrakrAI address, then **Runtime → Run all**. The cell prints an approval link and a code. Open the link and click **Approve** (the request also appears under **Model training → Colab GPU** in TrakrAI, where the code can be typed by hand). The cell then connects by itself. A request expires after 15 minutes; run the cell again to make a new one.

If you already created a pairing code in TrakrAI, paste it into `PAIRING_CODE` instead and no approval is needed.

The cell keeps running while the worker is connected; its log appears underneath.

**Stop it**

Press the stop button on the cell, or **Runtime → Disconnect and delete runtime**. A job that was running is handed back to TrakrAI and picked up again later. Your compute-unit usage is shown under **Runtime → View resources** and in Colab's billing page; TrakrAI shows which jobs this worker ran and for how long.

In [ ]:
# @title Connect this runtime to TrakrAI
TRAKRAI_URL = "https://ai.trakr.live"  # @param {type:"string"}
PAIRING_CODE = ""  # @param {type:"string"}
JOB_SLOTS = "train_detector:1,extract_frames:1,pretag_batch:1"  # @param {type:"string"}

import io
import json
import os
import shutil
import sys
import tarfile
import time
import urllib.error
import urllib.request
from pathlib import Path

TRAKRAI_URL = TRAKRAI_URL.strip().rstrip("/")
ROOT = Path(os.environ.get("TRAKRAI_COLAB_ROOT", "/content/trakrai"))
PKG = ROOT / "pkg"
STATE = ROOT / "worker.json"
ROOT.mkdir(parents=True, exist_ok=True)

if shutil.which("nvidia-smi") is None and not os.environ.get("TRAKRAI_ALLOW_NO_GPU"):
    raise SystemExit(
        "No GPU on this runtime. Choose Runtime > Change runtime type > T4 GPU and run again."
    )


def call(method, path, body=None, headers=None, timeout=60):
    data = None if body is None else json.dumps(body).encode()
    request = urllib.request.Request(
        TRAKRAI_URL + path,
        data=data,
        headers={"Content-Type": "application/json", **(headers or {})},
        method=method,
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read() or b"{}")
    except urllib.error.HTTPError as error:
        detail = error.read().decode(errors="replace")
        try:
            detail = json.loads(detail).get("error", detail)
        except ValueError:
            pass
        raise SystemExit(f"TrakrAI said: {detail}") from None


def pair_with_code(code):
    return call("POST", "/api/ml-worker/pair", {"code": code, "hostname": os.uname().nodename})


def pair_with_approval():
    requested = call("POST", "/api/ml-worker/pair/request", {"hostname": os.uname().nodename})
    print("=" * 72)
    print("Approve this runtime in TrakrAI (the link works for 15 minutes):")
    print()
    print(f"    {requested['approveUrl']}")
    print()
    print(f"    Code: {requested['code']}")
    print()
    print("Waiting for approval... (stop the cell to give up)")
    print("=" * 72, flush=True)
    headers = {"x-pairing-secret": requested["pollSecret"]}
    interval = max(2, int(requested.get("intervalSeconds", 5)))
    outcomes = {
        "denied": "The request was denied in TrakrAI. Run the cell again to ask again.",
        "expired": (
            "Nobody approved the request within 15 minutes. Run the cell again to ask again."
        ),
        "consumed": (
            "This request was already used. Run the cell again to make a new one."
        ),
    }
    while True:
        time.sleep(interval)
        result = call("GET", "/api/ml-worker/pair/poll", headers=headers)
        status = result.get("status")
        if status == "approved":
            return result
        if status != "pending":
            raise SystemExit(outcomes.get(status, f"Unexpected pairing status: {status}"))


token = json.loads(STATE.read_text())["token"] if STATE.exists() else None
if token is None:
    paired = pair_with_code(PAIRING_CODE.strip()) if PAIRING_CODE.strip() else pair_with_approval()
    token = paired["token"]
    STATE.write_text(json.dumps({"token": token, "name": paired["name"]}))
    STATE.chmod(0o600)
    print(f"Paired as {paired['name']} for {paired['factoryName']}.", flush=True)

request = urllib.request.Request(
    TRAKRAI_URL + "/api/ml-worker/package", headers={"Authorization": "Bearer " + token}
)
try:
    with urllib.request.urlopen(request, timeout=300) as response:
        archive = tarfile.open(fileobj=io.BytesIO(response.read()), mode="r:gz")
except urllib.error.HTTPError:
    STATE.unlink(missing_ok=True)
    raise SystemExit(
        "This worker's token is no longer valid. Run the cell again to connect it afresh."
    ) from None
shutil.rmtree(PKG, ignore_errors=True)
try:
    archive.extractall(PKG, filter="data")
except TypeError:
    archive.extractall(PKG)
sys.path.insert(0, str(PKG))

from trakrai_ml_worker.colab import run_worker

exit_code = run_worker(TRAKRAI_URL, token, PKG, slots=JOB_SLOTS, workdir=str(ROOT / "work"))
print(f"Worker stopped (exit code {exit_code}).")
if exit_code == 3:
    STATE.unlink(missing_ok=True)